In [1]:
!pip install -q uv
!uv python install 3.10
!uv venv --clear --python 3.10 /content/py312
!uv pip install -q --python /content/py312/bin/python numpy==1.26.4 pandas==2.2.3 scipy==1.15.3 scikit-learn==1.7.2 xgboost==2.1.4 lightgbm==4.7.0 joblib==1.4.2
!/content/py312/bin/python -c "import sys, numpy, scipy, sklearn, xgboost, lightgbm, pandas, joblib; print(sys.version.split()[0], numpy.__version__, scipy.__version__, sklearn.__version__, xgboost.__version__, lightgbm.__version__, pandas.__version__, joblib.__version__)"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.4/20.4 MB 75.5 MB/s eta 0:00:00
Installed Python 3.10.21 in 1.74s
 + cpython-3.10.21-linux-x86_64-gnu (python3.10)
Using CPython 3.10.21
Creating virtual environment at: py312
Activate with: source py312/bin/activate
3.10.21 1.26.4 1.15.3 1.7.2 2.1.4 4.7.0 2.2.3 1.4.2


In [2]:
import os
from google.colab import files
os.makedirs("/content/ml_training", exist_ok=True); os.chdir("/content/ml_training")
CSV = "nhamcs_2022_camp_relabeled_v2.csv"
if not os.path.exists(CSV):
    uploaded = files.upload()
    names = [n for n in uploaded if n.lower().endswith(".csv")]
    assert names, "No .csv file was uploaded."
    if names[0] != CSV: os.replace(names[0], CSV)
print(CSV, f"({os.path.getsize(CSV)/1e6:.1f} MB)")

Saving nhamcs_2022_camp_relabeled_v2.csv to nhamcs_2022_camp_relabeled_v2.csv
nhamcs_2022_camp_relabeled_v2.csv (3.3 MB)


In [3]:
code = r'''
"""severity_pipeline.py - SDRDCS Subsystem 3: features, 5-model stack, decision policy (training AND backend)."""
from __future__ import annotations
import re
import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix, hstack
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS, CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import MaxAbsScaler
from sklearn.utils.class_weight import compute_sample_weight

LABELS = ["LOW", "MEDIUM", "HIGH"]
LABEL_MAP = {"LOW": 0, "MEDIUM": 1, "HIGH": 2}
KNOWN_FLAGS = ["breathing_difficulty", "chest_pain", "focal_neurologic_deficit",
               "active_bleeding", "syncope", "seizure"]
VITAL_COLS = ["vital_hr", "vital_spo2", "vital_sbp", "vital_rr", "vital_temp", "pain_score"]
VITAL_THRESHOLDS = {
    "vf_bradycardia": ("vital_hr", "<", 50), "vf_hypox_severe": ("vital_spo2", "<", 90),
    "vf_hypox_moderate": ("vital_spo2", "<", 94), "vf_hypotension": ("vital_sbp", "<", 90),
    "vf_hypertension_crit": ("vital_sbp", ">", 180), "vf_tachypnea_severe": ("vital_rr", ">", 24),
    "vf_bradypnea": ("vital_rr", "<", 10), "vf_hypothermia": ("vital_temp", "<", 35.0),
    "vf_high_fever": ("vital_temp", ">", 38.5), "vf_pain_severe": ("pain_score", ">=", 8),
    "vf_pain_none": ("pain_score", "==", 0),
}
HIGH_COMPLAINT_PATTERNS = [
    "crush injury", "head trauma", "heart failure", "pregnancy complication", "traumatic brain injury",
    "head injury", "drowning", "electric shock", "gunshot wound", "stab wound", "chest trauma",
    "spinal injury", "paralysis weakness one side", "stroke", "seizure convulsion", "anaphylaxis"]
_KEEP = {"of", "in", "and", "or", "not", "no", "with", "without", "upper", "lower",
         "left", "right", "bilateral", "acute", "chronic"}
_STOP = [w for w in ENGLISH_STOP_WORDS if w not in _KEEP]
'''
with open("/content/ml_training/severity_pipeline.py", "w", encoding="utf-8") as f:
    f.write(code.strip("\n") + "\n\n\n")
print("part 1/6 ")

part 1/6 


In [4]:
code = r'''
def clean_text(text) -> str:
    if not isinstance(text, str):
        return ""
    text = re.sub(r"symptoms:\s*", "", text.lower())
    text = re.sub(r"[^a-z0-9_ ]", " ", text.replace("|", " "))
    return re.sub(r"\s+", " ", text).strip()


def split_complaints(text):
    if not isinstance(text, str):
        return []
    parts = [re.sub(r"\s+", " ", re.sub(r"[^a-z0-9_ ]", " ", p.lower())).strip() for p in text.split("|")]
    return [p for p in parts if p]


class FeatureBuilder:
    """fit() on TRAIN ONLY. Missing vitals -> train-only medians + miss_* indicators (no labels used)."""

    def __init__(self, max_tfidf_features=8000, min_complaint_count=3):
        self.max_tfidf_features, self.min_complaint_count = max_tfidf_features, min_complaint_count

    def fit(self, df):
        text = df["symptoms"].map(clean_text)
        self.tfidf_ = TfidfVectorizer(ngram_range=(1, 2), min_df=2, max_df=0.90,
                                      max_features=self.max_tfidf_features, sublinear_tf=True,
                                      stop_words=_STOP).fit(text)
        self.complaint_vec_ = CountVectorizer(tokenizer=split_complaints, lowercase=False, token_pattern=None,
                                              binary=True, min_df=self.min_complaint_count).fit(df["symptoms"])
        self.age_median_ = float(pd.to_numeric(df["age"], errors="coerce").median())
        self.vital_medians_ = {c: float(pd.to_numeric(df[c], errors="coerce").median()) for c in VITAL_COLS}
        dense, self.dense_names_ = self._dense(df)
        self.scaler_ = MaxAbsScaler().fit(hstack([self.tfidf_.transform(text), csr_matrix(dense)]).tocsr())
        return self
'''
with open("/content/ml_training/severity_pipeline.py", "a", encoding="utf-8") as f:
    f.write(code.strip("\n") + "\n\n\n")
print("part 2/6 ")

part 2/6 


In [5]:
code = r'''
    def _dense(self, df):
        cols, names = [], []

        def add(name, values):
            names.append(name); cols.append(np.asarray(values, dtype="float32").reshape(-1))

        age = pd.to_numeric(df["age"], errors="coerce").fillna(getattr(self, "age_median_", 35.0)).clip(0, 110)
        add("age_n", age / 100.0); add("age_infant", age < 1); add("age_child", age < 5)
        add("age_elderly", age >= 65); add("age_very_old", age >= 80)
        sym = df["symptoms"].fillna("").astype(str)
        add("n_complaints", sym.map(lambda s: len(split_complaints(s))))
        add("n_words", sym.map(lambda s: len(clean_text(s).split())))
        rf_total = np.zeros(len(df), dtype="float32")
        for f in KNOWN_FLAGS:
            v = pd.to_numeric(df[f"rf_{f}"], errors="coerce").fillna(0).values if f"rf_{f}" in df else np.zeros(len(df))
            add(f"rf_{f}", v); rf_total += (v > 0).astype("float32")
        has_rf = pd.to_numeric(df["has_red_flag"], errors="coerce").fillna(0).values if "has_red_flag" in df else rf_total
        add("has_red_flag", (np.asarray(has_rf) > 0) | (rf_total > 0)); add("red_flag_count", rf_total)
        raw = {c: (pd.to_numeric(df[c], errors="coerce") if c in df else pd.Series(np.nan, index=df.index))
               for c in VITAL_COLS}
        for c in VITAL_COLS:
            add(f"miss_{c}", raw[c].isna())
        for c in VITAL_COLS:
            add(c, raw[c].fillna(self.vital_medians_.get(c, np.nan)))
        flags = {}
        for name, (col, op, val) in VITAL_THRESHOLDS.items():
            v = raw[col]
            m = {"<": v < val, ">": v > val, ">=": v >= val, "==": v == val}[op]
            flags[name] = (m & v.notna()).astype(float); add(name, flags[name])
        crit = ["vf_hypox_severe", "vf_hypotension", "vf_bradycardia", "vf_hypothermia"]
        danger = crit + ["vf_tachypnea_severe", "vf_bradypnea"]
        add("vf_critical_vital", pd.concat([flags[c] for c in crit], axis=1).max(axis=1))
        add("vf_any_danger", pd.concat([flags[c] for c in danger], axis=1).max(axis=1))
        return np.column_stack(cols).astype("float32"), names
'''
with open("/content/ml_training/severity_pipeline.py", "a", encoding="utf-8") as f:
    f.write(code.strip("\n") + "\n\n\n")
print("part 3/6 ")

part 3/6 


In [6]:
code = r'''
    def transform(self, df):
        df = df.reset_index(drop=True)
        dense, _ = self._dense(df)
        X_lin = self.scaler_.transform(
            hstack([self.tfidf_.transform(df["symptoms"].map(clean_text)), csr_matrix(dense)]).tocsr()).astype("float32")
        X_tree = np.hstack([dense, self.complaint_vec_.transform(df["symptoms"]).toarray()]).astype("float32")
        return X_lin, X_tree


BASE_NAMES = ["lr_a", "lr_b", "lr_c", "lgb", "xgb"]
BASE_INPUT = {"lr_a": "lin", "lr_b": "lin", "lr_c": "lin", "lgb": "tree", "xgb": "tree"}


def make_base(name, seed=42):
    if name.startswith("lr_"):
        C = {"lr_a": 0.3, "lr_b": 0.1, "lr_c": 0.05}[name]
        return LogisticRegression(C=C, max_iter=2000, class_weight="balanced", solver="lbfgs", random_state=seed)
    if name == "lgb":
        import lightgbm as lgb
        return lgb.LGBMClassifier(n_estimators=400, learning_rate=0.05, num_leaves=31, min_child_samples=20,
                                  subsample=0.8, subsample_freq=1, colsample_bytree=0.8, reg_lambda=1.0,
                                  class_weight="balanced", random_state=seed, verbosity=-1, n_jobs=-1)
    import xgboost as xgb
    return xgb.XGBClassifier(n_estimators=400, learning_rate=0.05, max_depth=5, min_child_weight=3, gamma=0.1,
                             subsample=0.8, colsample_bytree=0.8, reg_lambda=1.0, objective="multi:softprob",
                             eval_metric="mlogloss", tree_method="hist", random_state=seed, verbosity=0, n_jobs=-1)


def fit_base(name, X_lin, X_tree, y, seed=42):
    model, X = make_base(name, seed), (X_lin if BASE_INPUT[name] == "lin" else X_tree)
    if name == "xgb":
        model.fit(X, y, sample_weight=compute_sample_weight("balanced", y))
    else:
        model.fit(X, y)
    return model


def base_probabilities(models, X_lin, X_tree):
    return np.hstack([models[n].predict_proba(X_lin if BASE_INPUT[n] == "lin" else X_tree) for n in BASE_NAMES])


def meta_features(base_proba):
    return np.log(np.clip(base_proba, 1e-6, 1.0))
'''
with open("/content/ml_training/severity_pipeline.py", "a", encoding="utf-8") as f:
    f.write(code.strip("\n") + "\n\n\n")
print("part 4/6 ")

part 4/6 


In [7]:
code = r'''
def threshold_cascade(proba, t_high, t_low):
    pred = np.argmax(proba, axis=1).copy()
    pred[proba[:, 2] >= t_high] = 2
    pred[(proba[:, 0] >= t_low) & (proba[:, 2] < t_high)] = 0
    return pred


_COMPLAINT_RE = re.compile("|".join(re.escape(p) for p in HIGH_COMPLAINT_PATTERNS), re.I)


def safety_masks(df):
    df = df.reset_index(drop=True)
    rf = np.zeros(len(df), dtype=bool)
    if "has_red_flag" in df:
        rf |= pd.to_numeric(df["has_red_flag"], errors="coerce").fillna(0).values > 0
    for f in KNOWN_FLAGS:
        if f"rf_{f}" in df:
            rf |= pd.to_numeric(df[f"rf_{f}"], errors="coerce").fillna(0).values > 0
    return {"red_flag": rf, "complaint": df["symptoms"].fillna("").astype(str).str.contains(_COMPLAINT_RE).values}


def apply_safety_layers(pred, df, layers=("red_flag", "complaint")):
    pred, masks = pred.copy(), safety_masks(df)
    for layer in layers:
        pred[masks[layer]] = 2          # floor to HIGH; never lowers a prediction
    return pred
'''
with open("/content/ml_training/severity_pipeline.py", "a", encoding="utf-8") as f:
    f.write(code.strip("\n") + "\n\n\n")
print("part 5/6")

part 5/6


In [8]:
code = r'''
class StackBundle:
    """Everything the backend needs: features + 5 base models (full-train) + meta-learner + thresholds."""

    def __init__(self, features, base_models, meta, t_high, t_low, layers=("red_flag", "complaint")):
        self.features, self.base_models, self.meta = features, base_models, meta
        self.t_high, self.t_low, self.layers = t_high, t_low, tuple(layers)

    def predict_proba(self, df):
        X_lin, X_tree = self.features.transform(df)
        return self.meta.predict_proba(meta_features(base_probabilities(self.base_models, X_lin, X_tree)))

    def predict(self, df, use_layers=True):
        proba = self.predict_proba(df)
        model_pred = threshold_cascade(proba, self.t_high, self.t_low)
        final = apply_safety_layers(model_pred, df, self.layers) if use_layers else model_pred
        return {"proba": proba, "model_pred": model_pred, "final_pred": final,
                "final_labels": [LABELS[i] for i in final], "layer_masks": safety_masks(df)}


def row_from_request(symptoms, age=None, rf_flags=None, vitals=None):
    """One-row DataFrame for a classify request (symptoms like 'Chest pain | Cough')."""
    rf_flags = rf_flags or {}
    row = {"symptoms": symptoms or "", "age": np.nan if age is None else float(age)}
    for f in KNOWN_FLAGS:
        row[f"rf_{f}"] = int(bool(rf_flags.get(f, 0)))
    row["red_flag_count"] = sum(row[f"rf_{f}"] for f in KNOWN_FLAGS)
    row["has_red_flag"] = int(row["red_flag_count"] > 0)
    for c in VITAL_COLS:
        row[c] = (vitals or {}).get(c, np.nan)
    return pd.DataFrame([row])


def load_bundle(path):
    import sys, joblib
    sys.modules.setdefault("severity_pipeline", sys.modules[__name__])   # works if imported under another path
    return joblib.load(path)
'''
with open("/content/ml_training/severity_pipeline.py", "a", encoding="utf-8") as f:
    f.write(code.strip("\n") + "\n\n\n")
print("part 6/6 ")

part 6/6 


In [9]:
import py_compile
p = "/content/ml_training/severity_pipeline.py"
n = len(open(p, encoding="utf-8").read().splitlines())
assert n == 215, f"severity_pipeline.py has {n} lines, expected 215. A part was skipped, pasted twice or edited: re-run Cell 3 (it resets the file), then Cells 4-8 once each, in order."
py_compile.compile(p, doraise=True)
print(f"severity_pipeline.py OK ({n} lines)")

severity_pipeline.py OK (215 lines)


In [10]:
code = r'''
# train_run.py - trains the SDRDCS severity model with the backend library versions
import os
os.chdir(os.path.dirname(os.path.abspath(__file__)))

# CELL 2 - imports, config, version check (no Google Drive: everything lives in the Colab session folder)
import os, sys, json, time, warnings, shutil
import numpy as np, pandas as pd, sklearn, xgboost, lightgbm, joblib
warnings.filterwarnings("ignore")

STRICT_VERSIONS = True      # False: train with the library versions Colab already has (fast, no install needed)
                            # True : require the pinned versions below (needed if a backend must load the saved model)
WORK = os.getcwd()                       # Colab: /content
SEED, N_SPLITS, HIGH_RECALL_FLOOR = 42, 5, 0.90
LAYERS = ("red_flag", "complaint")
DATA_FILE, LABEL_COL, GROUP_COL = "nhamcs_2022_camp_relabeled_v2.csv", "camp_triage_label_final", "model_input_text"
OUT = os.path.join(WORK, "model_artefacts", "my_run"); os.makedirs(OUT, exist_ok=True)

ACTUAL = {"numpy": np.__version__, "sklearn": sklearn.__version__, "xgboost": xgboost.__version__, "lightgbm": lightgbm.__version__}
EXPECTED = {"numpy": "1.26.4", "sklearn": "1.7.2", "xgboost": "2.1.4", "lightgbm": "4.7.0"}
print("Python", sys.version.split()[0], "|", ACTUAL)
if ACTUAL != EXPECTED:
    msg = f"Versions differ from the backend pins {EXPECTED}."
    assert not STRICT_VERSIONS, msg + " Re-run Cell 1 and then Cell 16."
    print("NOTE:", msg, "Training works as normal, but a model saved now can only be loaded by a backend with these same versions.")

# the CSV must sit next to this script
assert os.path.exists(DATA_FILE), f"{DATA_FILE} not found in {os.getcwd()}"
print("Dataset:", DATA_FILE, f"({os.path.getsize(DATA_FILE)/1e6:.1f} MB)")
'''
with open("/content/ml_training/train_run.py", "w", encoding="utf-8") as f:
    f.write(code.strip("\n") + "\n\n\n")
print("part 1/5 ")

part 1/5 


In [11]:
code = r'''
# CELL 10 - check the module file and import it
import importlib, py_compile
MODULE_PATH = os.path.join(WORK, "severity_pipeline.py")
n_lines = len(open(MODULE_PATH, encoding="utf-8").read().splitlines())
assert n_lines == 215, (f"severity_pipeline.py has {n_lines} lines, expected 215. A part was skipped, pasted twice or "
                                   "edited: re-run Cell 3 (it resets the file), then Cells 4-8 once each, in order.")
py_compile.compile(MODULE_PATH, doraise=True)
sys.modules.pop("severity_pipeline", None); importlib.invalidate_caches(); sys.path.insert(0, WORK)
import severity_pipeline as sp
need = ["FeatureBuilder", "StackBundle", "fit_base", "base_probabilities", "meta_features",
        "threshold_cascade", "apply_safety_layers", "row_from_request", "load_bundle"]
assert all(hasattr(sp, n) for n in need), "module is incomplete"
print(f"severity_pipeline.py OK ({n_lines} lines)")

# CELL 11 - load data, leakage guard, leak-free split
from sklearn.model_selection import StratifiedGroupKFold

df = pd.read_csv(DATA_FILE, low_memory=False)
INPUT_COLS = (["symptoms", "age", "has_red_flag", "red_flag_count"] + [f"rf_{f}" for f in sp.KNOWN_FLAGS] + sp.VITAL_COLS)
missing = [c for c in INPUT_COLS + [LABEL_COL, GROUP_COL] if c not in df.columns]
assert not missing, f"This CSV is missing required columns: {missing}"
df = df[df[LABEL_COL].isin(sp.LABEL_MAP)].reset_index(drop=True)
df["y"] = df[LABEL_COL].map(sp.LABEL_MAP).astype(int)
print(f"{len(df):,} patients"); print(df[LABEL_COL].value_counts().reindex(sp.LABELS).to_string())
assert not [c for c in INPUT_COLS if c.startswith(("dx_", "camp_triage", "audit_priority")) or c in ("target", "y", GROUP_COL)]

# 80/10/10, stratified by class, grouped by patient profile (same complaints+age never straddles splits)
folds = list(StratifiedGroupKFold(10, shuffle=True, random_state=SEED).split(df, df["y"], groups=df[GROUP_COL]))
idx_test, idx_val = folds[0][1], folds[1][1]
idx_train = np.setdiff1d(np.arange(len(df)), np.r_[idx_test, idx_val])
tr, va, te = (df.iloc[i].reset_index(drop=True) for i in (idx_train, idx_val, idx_test))
tr_in, va_in, te_in = tr[INPUT_COLS], va[INPUT_COLS], te[INPUT_COLS]
y_tr, y_va, y_te = tr["y"].values, va["y"].values, te["y"].values
assert not (set(tr[GROUP_COL]) & set(va[GROUP_COL])) and not (set(tr[GROUP_COL]) & set(te[GROUP_COL])) \
       and not (set(va[GROUP_COL]) & set(te[GROUP_COL])), "profile leaked across splits"
for n, y in [("TRAIN", y_tr), ("VAL", y_va), ("TEST", y_te)]:
    print(f"{n:6s} n={len(y):5d}  LOW/MED/HIGH = {np.bincount(y, minlength=3)}")
'''
with open("/content/ml_training/train_run.py", "a", encoding="utf-8") as f:
    f.write(code.strip("\n") + "\n\n\n")
print("part 2/5 ")

part 2/5 


In [12]:
code = r'''
# CELL 12 - features (TRAIN only) + 5-fold OOF stacking (features re-fit inside every fold)
from sklearn.metrics import balanced_accuracy_score
t0 = time.time()
fb = sp.FeatureBuilder().fit(tr_in)
X_lin_tr, X_tree_tr = fb.transform(tr_in)
print("linear matrix", X_lin_tr.shape, "| tree matrix", X_tree_tr.shape)

oof = np.zeros((len(tr), len(sp.BASE_NAMES) * 3))
sgkf = StratifiedGroupKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
for k, (a, b) in enumerate(sgkf.split(tr_in, y_tr, groups=tr[GROUP_COL])):
    fb_k = sp.FeatureBuilder().fit(tr_in.iloc[a])
    Xl_a, Xt_a = fb_k.transform(tr_in.iloc[a]); Xl_b, Xt_b = fb_k.transform(tr_in.iloc[b])
    for i, name in enumerate(sp.BASE_NAMES):
        m = sp.fit_base(name, Xl_a, Xt_a, y_tr[a], seed=SEED + k)
        oof[b, i*3:(i+1)*3] = m.predict_proba(Xl_b if sp.BASE_INPUT[name] == "lin" else Xt_b)
    print(f"  fold {k+1}/{N_SPLITS} done ({time.time()-t0:.0f}s)")
assert np.allclose(oof.reshape(len(tr), -1, 3).sum(-1), 1, atol=1e-4)
for i, name in enumerate(sp.BASE_NAMES):
    print(f"  OOF balanced accuracy {name:5s}: {balanced_accuracy_score(y_tr, oof[:, i*3:(i+1)*3].argmax(1)):.4f}")
'''
with open("/content/ml_training/train_run.py", "a", encoding="utf-8") as f:
    f.write(code.strip("\n") + "\n\n\n")
print("part 3/5 ")

part 3/5 


In [13]:
code = r'''
# CELL 13 - meta-learner and decision thresholds (chosen on TRAIN out-of-fold predictions only)
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_predict

base_full = {n: sp.fit_base(n, X_lin_tr, X_tree_tr, y_tr, seed=SEED) for n in sp.BASE_NAMES}
F_oof = sp.meta_features(oof)
cv_meta = StratifiedGroupKFold(5, shuffle=True, random_state=SEED + 1)
best = (-1, None)
for C in [0.1, 1.0, 10.0]:
    p = cross_val_predict(LogisticRegression(C=C, max_iter=3000, class_weight="balanced", random_state=SEED),
                          F_oof, y_tr, cv=cv_meta, groups=tr[GROUP_COL], method="predict_proba")
    s = balanced_accuracy_score(y_tr, p.argmax(1)); print(f"  meta C={C:<5} grouped-CV balanced accuracy = {s:.4f}")
    if s > best[0]: best = (s, C, p)
_, best_C, meta_oof = best
meta = LogisticRegression(C=best_C, max_iter=3000, class_weight="balanced", random_state=SEED).fit(F_oof, y_tr)

def conf(y, p): return np.bincount(3 * np.asarray(y) + np.asarray(p), minlength=9).reshape(3, 3)
def rates(cm):
    tp = np.diag(cm).astype(float); rec = tp / np.maximum(cm.sum(1), 1); prec = tp / np.maximum(cm.sum(0), 1)
    return rec, 2 * prec * rec / np.maximum(prec + rec, 1e-12)

layer_mask = np.zeros(len(tr), bool)
for l in LAYERS: layer_mask |= sp.safety_masks(tr_in)[l]
grid = []
for th in np.arange(0.20, 0.81, 0.01):
    for tl in np.arange(0.30, 0.81, 0.01):
        pred = sp.threshold_cascade(meta_oof, th, tl); pred[layer_mask] = 2
        rec, _ = rates(conf(y_tr, pred)); grid.append((th, tl, rec.mean(), rec[2]))
grid = pd.DataFrame(grid, columns=["th", "tl", "bal", "high_rec"])
ok = grid[grid.high_rec >= HIGH_RECALL_FLOOR]
row = (ok if len(ok) else grid.sort_values("high_rec", ascending=False)).sort_values("bal", ascending=False).iloc[0]
TH, TL = float(row.th), float(row.tl)
print(f"meta C={best_C}   T_HIGH={TH:.2f}  T_LOW={TL:.2f}   (TRAIN-OOF balanced acc {row.bal:.4f}, HIGH recall {row.high_rec:.4f})")
'''
with open("/content/ml_training/train_run.py", "a", encoding="utf-8") as f:
    f.write(code.strip("\n") + "\n\n\n")
print("part 4/5 ")

part 4/5 


In [17]:
code = r'''
# CELL 14 - evaluate TRAIN (fitted + OOF), VALIDATION, TEST: reports, confusion matrices (printed), accuracy table, 80 % check
from sklearn.metrics import classification_report
bundle = sp.StackBundle(fb, base_full, meta, TH, TL, layers=LAYERS)
pred = {"train_fitted": bundle.predict(tr_in)["final_pred"],
        "train_oof": sp.apply_safety_layers(sp.threshold_cascade(meta_oof, TH, TL), tr_in, LAYERS),
        "val": bundle.predict(va_in)["final_pred"], "test": bundle.predict(te_in)["final_pred"]}
Y = {"train_fitted": y_tr, "train_oof": y_tr, "val": y_va, "test": y_te}
NAME = {"train_fitted": "TRAIN (fitted - optimistic, do not report)", "train_oof": "TRAIN (5-fold OOF - honest)",
        "val": "VALIDATION", "test": "TEST"}
metrics = {}
for k in ["train_fitted", "train_oof", "val", "test"]:
    cm = conf(Y[k], pred[k]); rec, f1 = rates(cm)
    metrics[k] = {"n": int(cm.sum()), "accuracy": float(np.trace(cm) / cm.sum()), "bal_acc": float(rec.mean()),
                  "macro_f1": float(f1.mean()), "high_recall": float(rec[2]), "high_to_low": int(cm[2, 0]), "cm": cm.tolist()}
    print("=" * 66); print(f"  {NAME[k]}   (n={cm.sum()})"); print("=" * 66)
    print(classification_report(Y[k], pred[k], target_names=sp.LABELS, digits=3, zero_division=0))
    print(pd.DataFrame(cm, index=[f"true {l}" for l in sp.LABELS], columns=[f"pred {l}" for l in sp.LABELS]).to_string())
    print(f"\n  ACCURACY = {metrics[k]['accuracy']:.4f}   balanced acc = {metrics[k]['bal_acc']:.4f}   "
          f"HIGH->LOW errors = {metrics[k]['high_to_low']}\n")

summary = (pd.DataFrame(metrics).T.drop(columns="cm")
           .astype({"n": int, "accuracy": float, "bal_acc": float, "macro_f1": float, "high_recall": float, "high_to_low": int}).round(4))
print(summary.to_string())

print("\n>>> 80 % CHECK (held-out data only; nothing is capped, so values above 90 % are simply reported)")
for k in ["val", "test"]:
    for m_ in ["accuracy", "bal_acc", "macro_f1", "high_recall"]:
        v = metrics[k][m_]; print(f"  {k:5s} {m_:12s} {v:.4f}   {'PASS (>= 80 %)' if v >= 0.80 else 'FAIL (< 80 %)'}")

# CELL 15 - save the model, prove it reloads identically, and download everything to your device
bundle_path = os.path.join(OUT, "severity_bundle.joblib")
joblib.dump(bundle, bundle_path, compress=3)
loaded = sp.load_bundle(bundle_path)
assert np.allclose(loaded.predict(te_in)["proba"], bundle.predict(te_in)["proba"], atol=1e-9)
assert (loaded.predict(te_in)["final_pred"] == pred["test"]).all()
with open(os.path.join(OUT, "metrics.json"), "w") as f:
    json.dump({"libraries": ACTUAL, "thresholds": [TH, TL], "meta_C": best_C, "layers": list(LAYERS), "metrics": metrics}, f, indent=2)
shutil.copy(MODULE_PATH, OUT)
zip_path = shutil.make_archive(os.path.join(WORK, "severity_model_run"), "zip", OUT)
print("Saved + reload verified:", sorted(os.listdir(OUT)))
try:
    from google.colab import files
    files.download(zip_path)             # Colab sessions are temporary: this keeps a copy on your device
except ImportError:
    print("Zip created at", zip_path)

'''
with open("/content/ml_training/train_run.py", "a", encoding="utf-8") as f:
    f.write(code.strip("\n") + "\n\n\n")
print("part 5/5 ")

part 5/5 


In [20]:
p = "/content/ml_training/train_run.py"
lines = open(p, encoding="utf-8").read().splitlines()
print("lines before:", len(lines))
with open(p, "w", encoding="utf-8") as f:
    f.write("\n".join(lines[:173]) + "\n")
print("lines after :", len(open(p, encoding="utf-8").read().splitlines()))

lines before: 220
lines after : 173


In [21]:
p = "/content/ml_training/train_run.py"
n = len(open(p, encoding="utf-8").read().splitlines())
assert n == 173, f"train_run.py has {n} lines, expected 173. A part was skipped, pasted twice or edited: re-run Cell 10 (it resets the file), then Cells 11-14 once each, in order."
print(f"train_run.py OK ({n} lines)")

train_run.py OK (173 lines)


In [22]:
!/content/py312/bin/python /content/ml_training/train_run.py

Python 3.10.21 | {'numpy': '1.26.4', 'sklearn': '1.7.2', 'xgboost': '2.1.4', 'lightgbm': '4.7.0'}
Dataset: nhamcs_2022_camp_relabeled_v2.csv (3.3 MB)
severity_pipeline.py OK (215 lines)
7,578 patients
camp_triage_label_final
LOW       3124
MEDIUM    2472
HIGH      1982
TRAIN  n= 6090  LOW/MED/HIGH = [2516 1973 1601]
VAL    n=  748  LOW/MED/HIGH = [320 258 170]
TEST   n=  740  LOW/MED/HIGH = [288 241 211]
linear matrix (6090, 1457) | tree matrix (6090, 193)
  fold 1/5 done (9s)
  fold 2/5 done (17s)
  fold 3/5 done (23s)
  fold 4/5 done (31s)
  fold 5/5 done (37s)
  OOF balanced accuracy lr_a : 0.8787
  OOF balanced accuracy lr_b : 0.8611
  OOF balanced accuracy lr_c : 0.8444
  OOF balanced accuracy lgb  : 0.9233
  OOF balanced accuracy xgb  : 0.9299
  meta C=0.1   grouped-CV balanced accuracy = 0.9420
  meta C=1.0   grouped-CV balanced accuracy = 0.9412
  meta C=10.0  grouped-CV balanced accuracy = 0.9411
meta C=0.1   T_HIGH=0.39  T_LOW=0.40   (TRAIN-OOF balanced acc 0.9158, HIGH recal

In [23]:
import os
from google.colab import files
zip_path = "/content/ml_training/severity_model_run.zip"
assert os.path.exists(zip_path), "Training did not finish - check the output of the previous cell"
files.download(zip_path)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>